# Create Spark Session

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("OrderProcessingPipeline") \
    .getOrCreate()


In [3]:
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=False)

# Print the Schema

In [4]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)



# Count total Rows

In [5]:
orders_df.count()

300000

# Dataset Sample

In [6]:
orders_df.show(10, truncate=False)

+-----------+-----------+-----------+-----------+-----------+-------+----------+---------+
|order_id   |customer_id|city       |category   |product    |amount |order_date|status   |
+-----------+-----------+-----------+-----------+-----------+-------+----------+---------+
|ORD00000000|C000000    | hyderabad | grocery   |Oil        |invalid|01/01/2024|Cancelled|
|ORD00000001|C000001    |Pune       |Grocery    |Sugar      |35430  |2024-01-02|Completed|
|ORD00000002|C000002    |Pune       |Electronics|Mobile     |65358  |2024-01-03|Completed|
|ORD00000003|C000003    |Bangalore  |Electronics|Laptop     |5558   |2024-01-04|Completed|
|ORD00000004|C000004    |Pune       |Home       |AirPurifier|33659  |2024-01-05|Completed|
|ORD00000005|C000005    |Delhi      |Fashion    |Jeans      |8521   |2024-01-06|Completed|
|ORD00000006|C000006    |Delhi      |Grocery    |Sugar      |42383  |2024-01-07|Completed|
|ORD00000007|C000007    |Pune       |Grocery    |Rice       |45362  |2024-01-08|Completed|

#Why treat all columns as StringType initially?

*   Data Quality Issues: Amount column has invalid values like "invalid", "12,000", empty strings. If we infer schema, Spark might fail or misinterpret these.
*   Mixed Formats: Dates have multiple formats (e.g., 2024-01-15, 15/01/2024), which cannot be parsed into a single DateType automatically.



# Remove Trailing Spaces

In [7]:
from pyspark.sql import functions as F

cols_to_trim = ["city", "category", "product"]

cleaned_df = orders_df
for c in cols_to_trim:
    cleaned_df = cleaned_df.withColumn(c, F.trim(F.col(c)))


In [8]:
cleaned_df.show(10, truncate=False)

+-----------+-----------+---------+-----------+-----------+-------+----------+---------+
|order_id   |customer_id|city     |category   |product    |amount |order_date|status   |
+-----------+-----------+---------+-----------+-----------+-------+----------+---------+
|ORD00000000|C000000    |hyderabad|grocery    |Oil        |invalid|01/01/2024|Cancelled|
|ORD00000001|C000001    |Pune     |Grocery    |Sugar      |35430  |2024-01-02|Completed|
|ORD00000002|C000002    |Pune     |Electronics|Mobile     |65358  |2024-01-03|Completed|
|ORD00000003|C000003    |Bangalore|Electronics|Laptop     |5558   |2024-01-04|Completed|
|ORD00000004|C000004    |Pune     |Home       |AirPurifier|33659  |2024-01-05|Completed|
|ORD00000005|C000005    |Delhi    |Fashion    |Jeans      |8521   |2024-01-06|Completed|
|ORD00000006|C000006    |Delhi    |Grocery    |Sugar      |42383  |2024-01-07|Completed|
|ORD00000007|C000007    |Pune     |Grocery    |Rice       |45362  |2024-01-08|Completed|
|ORD00000008|C000008 

# Convert to Proper Case

In [9]:

from pyspark.sql import functions as F

standardized_df = (
    cleaned_df
    .withColumn("city", F.initcap(F.col("city")))
    .withColumn("category", F.initcap(F.col("category")))
    .withColumn("product", F.initcap(F.col("product")))
)

standardized_df.select("city", "category", "product").show(20, truncate=False)

+---------+-----------+-----------+
|city     |category   |product    |
+---------+-----------+-----------+
|Hyderabad|Grocery    |Oil        |
|Pune     |Grocery    |Sugar      |
|Pune     |Electronics|Mobile     |
|Bangalore|Electronics|Laptop     |
|Pune     |Home       |Airpurifier|
|Delhi    |Fashion    |Jeans      |
|Delhi    |Grocery    |Sugar      |
|Pune     |Grocery    |Rice       |
|Bangalore|Fashion    |Jeans      |
|Kolkata  |Electronics|Laptop     |
|Bangalore|Grocery    |Sugar      |
|Kolkata  |Electronics|Tablet     |
|Bangalore|Grocery    |Sugar      |
|Pune     |Fashion    |Tshirt     |
|Mumbai   |Electronics|Tablet     |
|Pune     |Electronics|Mobile     |
|Mumbai   |Home       |Mixer      |
|Bangalore|Grocery    |Oil        |
|Kolkata  |Fashion    |Jeans      |
|Mumbai   |Electronics|Mobile     |
+---------+-----------+-----------+
only showing top 20 rows


# Standardize the Data

In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

amount_norm = F.regexp_replace(F.trim(F.col("amount")), r",", "")

tmp_df = (
    standardized_df
    .withColumn("amount_norm",
                F.when(F.length(amount_norm) == 0, F.lit(None)).otherwise(amount_norm))
)


In [11]:
valid_int = F.col("amount_norm").rlike(r"^[+-]?\d+$")

cleaned_amount_df = (
    tmp_df
    .withColumn(
        "amount",
        F.when(valid_int, F.col("amount_norm").cast(IntegerType()))
         .otherwise(F.lit(None).cast(IntegerType()))
    )
    .drop("amount_norm")
)


In [12]:

cleaned_amount_df.select("amount").summary().show()
cleaned_amount_df.filter(F.col("amount").isNull()).select("order_id", "amount").show(20, truncate=False)


+-------+-----------------+
|summary|           amount|
+-------+-----------------+
|  count|           274836|
|   mean|43787.37508550554|
| stddev|26192.67426878039|
|    min|              500|
|    25%|            19732|
|    50%|            43117|
|    75%|            66707|
|    max|            90000|
+-------+-----------------+

+-----------+------+
|order_id   |amount|
+-----------+------+
|ORD00000000|NULL  |
|ORD00000019|NULL  |
|ORD00000029|NULL  |
|ORD00000038|NULL  |
|ORD00000057|NULL  |
|ORD00000058|NULL  |
|ORD00000076|NULL  |
|ORD00000087|NULL  |
|ORD00000095|NULL  |
|ORD00000114|NULL  |
|ORD00000116|NULL  |
|ORD00000133|NULL  |
|ORD00000145|NULL  |
|ORD00000152|NULL  |
|ORD00000171|NULL  |
|ORD00000174|NULL  |
|ORD00000190|NULL  |
|ORD00000203|NULL  |
|ORD00000209|NULL  |
|ORD00000228|NULL  |
+-----------+------+
only showing top 20 rows


In [13]:
cleaned_amount_df.show()

+-----------+-----------+---------+-----------+-----------+------+----------+---------+
|   order_id|customer_id|     city|   category|    product|amount|order_date|   status|
+-----------+-----------+---------+-----------+-----------+------+----------+---------+
|ORD00000000|    C000000|Hyderabad|    Grocery|        Oil|  NULL|01/01/2024|Cancelled|
|ORD00000001|    C000001|     Pune|    Grocery|      Sugar| 35430|2024-01-02|Completed|
|ORD00000002|    C000002|     Pune|Electronics|     Mobile| 65358|2024-01-03|Completed|
|ORD00000003|    C000003|Bangalore|Electronics|     Laptop|  5558|2024-01-04|Completed|
|ORD00000004|    C000004|     Pune|       Home|Airpurifier| 33659|2024-01-05|Completed|
|ORD00000005|    C000005|    Delhi|    Fashion|      Jeans|  8521|2024-01-06|Completed|
|ORD00000006|    C000006|    Delhi|    Grocery|      Sugar| 42383|2024-01-07|Completed|
|ORD00000007|    C000007|     Pune|    Grocery|       Rice| 45362|2024-01-08|Completed|
|ORD00000008|    C000008|Bangalo

In [17]:

from pyspark.sql import functions as F
date_str = F.trim(F.col("order_date"))

ts1 = F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")
ts2 = F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")
ts3 = F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")

order_date_clean = F.to_date(F.coalesce(ts1, ts2, ts3))

final_df = cleaned_amount_df.withColumn("order_date_clean", order_date_clean)

final_df.select("order_id", "order_date", "order_date_clean").show(20, truncate=False)
final_df.printSchema()

failed = final_df.filter(F.col("order_date").isNotNull() & F.col("order_date_clean").isNull()).count()
print(f"Unparseable order_date values: {failed}")

+-----------+----------+----------------+
|order_id   |order_date|order_date_clean|
+-----------+----------+----------------+
|ORD00000000|01/01/2024|2024-01-01      |
|ORD00000001|2024-01-02|2024-01-02      |
|ORD00000002|2024-01-03|2024-01-03      |
|ORD00000003|2024-01-04|2024-01-04      |
|ORD00000004|2024-01-05|2024-01-05      |
|ORD00000005|2024-01-06|2024-01-06      |
|ORD00000006|2024-01-07|2024-01-07      |
|ORD00000007|2024-01-08|2024-01-08      |
|ORD00000008|2024-01-09|2024-01-09      |
|ORD00000009|2024-01-10|2024-01-10      |
|ORD00000010|2024-01-11|2024-01-11      |
|ORD00000011|12/01/2024|2024-01-12      |
|ORD00000012|2024-01-13|2024-01-13      |
|ORD00000013|2024/01/14|2024-01-14      |
|ORD00000014|2024-01-15|2024-01-15      |
|ORD00000015|2024-01-16|2024-01-16      |
|ORD00000016|2024-01-17|2024-01-17      |
|ORD00000017|2024-01-18|2024-01-18      |
|ORD00000018|2024-01-19|2024-01-19      |
|ORD00000019|2024-01-20|2024-01-20      |
+-----------+----------+----------

# Count invalid nums

In [18]:

invalid_amount_count = final_df.filter(F.col("amount").isNull()).count()
date_str = F.trim(F.col("order_date"))
invalid_date_count = final_df.filter(
    (F.col("order_date_clean").isNull()) &
    (date_str.isNotNull()) &
    (F.length(date_str) > 0)
).count()


# Identify Duplicates

In [19]:
dups_df = (
    final_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc(), F.col("order_id").asc())
)


In [20]:
print("Duplicate order_id values (top 50 by frequency):")
dups_df.show(50, truncate=False)

Duplicate order_id values (top 50 by frequency):
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [21]:
dedup_df = final_df.dropDuplicates(["order_id"])
completed_df = dedup_df.filter(F.lower(F.trim(F.col("status"))) == F.lit("completed"))

In [23]:
completed_df.show()

+-----------+-----------+---------+-----------+-----------+------+----------+---------+----------------+
|   order_id|customer_id|     city|   category|    product|amount|order_date|   status|order_date_clean|
+-----------+-----------+---------+-----------+-----------+------+----------+---------+----------------+
|ORD00000001|    C000001|     Pune|    Grocery|      Sugar| 35430|2024-01-02|Completed|      2024-01-02|
|ORD00000007|    C000007|     Pune|    Grocery|       Rice| 45362|2024-01-08|Completed|      2024-01-08|
|ORD00000008|    C000008|Bangalore|    Fashion|      Jeans| 10563|2024-01-09|Completed|      2024-01-09|
|ORD00000010|    C000010|Bangalore|    Grocery|      Sugar| 66576|2024-01-11|Completed|      2024-01-11|
|ORD00000011|    C000011|  Kolkata|Electronics|     Tablet| 50318|12/01/2024|Completed|      2024-01-12|
|ORD00000012|    C000012|Bangalore|    Grocery|      Sugar| 84768|2024-01-13|Completed|      2024-01-13|
|ORD00000014|    C000014|   Mumbai|Electronics|     Tab

In [25]:
stage_counts = {}
stage_counts["start"] = final_df.count()
stage_counts["after_dedup"] = dedup_df.count()
stage_counts["after_completed_filter"] = completed_df.count()

In [26]:
duplicate_rows_removed = stage_counts["start"] - stage_counts["after_dedup"]

# Check the number of partitions

In [27]:
current_partitions = completed_df.rdd.getNumPartitions()
print(current_partitions)


2


In [28]:
shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")

# Total Revenue

In [29]:

from pyspark.sql import functions as F

revenue_by_city = (
    completed_df
    .groupBy("city")
    .agg(F.sum("amount").alias("total_revenue"))
)

revenue_by_city.show(20, truncate=False)

+---------+-------------+
|city     |total_revenue|
+---------+-------------+
|Bangalore|1628527093   |
|Chennai  |1629865247   |
|Mumbai   |1625518096   |
|Kolkata  |1624300497   |
|Pune     |1646196535   |
|Delhi    |1639639916   |
|Hyderabad|1642443340   |
+---------+-------------+



# Explain

In [30]:
revenue_by_city.explain(True)

== Parsed Logical Plan ==
'Aggregate ['city], ['city, 'sum('amount) AS total_revenue#773]
+- Filter (lower(trim(status#24, None)) = completed)
   +- Deduplicate [order_id#17]
      +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, amount#124, order_date#23, status#24, to_date(coalesce(try_to_timestamp(order_date#23, Some(yyyy-MM-dd), TimestampType, Some(Etc/UTC), false), try_to_timestamp(order_date#23, Some(dd/MM/yyyy), TimestampType, Some(Etc/UTC), false), try_to_timestamp(order_date#23, Some(yyyy/MM/dd), TimestampType, Some(Etc/UTC), false)), None, Some(Etc/UTC), true) AS order_date_clean#321]
         +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, amount#124, order_date#23, status#24]
            +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, CASE WHEN RLIKE(amount_norm#123, ^[+-]?\d+$) THEN cast(amount_norm#123 as int) ELSE cast(null as int) END AS amount#124, order_date#23, status#24, amount_

# Repartition the dataset by city

In [31]:
completed_df_by_city = completed_df.repartition("city")

In [32]:
after_repartition_parts = completed_df_by_city.rdd.getNumPartitions()

In [33]:
revenue_by_city_repart = (
    completed_df_by_city
    .groupBy("city")
    .agg(F.sum("amount").alias("total_revenue"))
)

In [34]:
revenue_by_city_repart.explain(True)

== Parsed Logical Plan ==
'Aggregate ['city], ['city, 'sum('amount) AS total_revenue#1002]
+- RepartitionByExpression [city#107]
   +- Filter (lower(trim(status#24, None)) = completed)
      +- Deduplicate [order_id#17]
         +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, amount#124, order_date#23, status#24, to_date(coalesce(try_to_timestamp(order_date#23, Some(yyyy-MM-dd), TimestampType, Some(Etc/UTC), false), try_to_timestamp(order_date#23, Some(dd/MM/yyyy), TimestampType, Some(Etc/UTC), false), try_to_timestamp(order_date#23, Some(yyyy/MM/dd), TimestampType, Some(Etc/UTC), false)), None, Some(Etc/UTC), true) AS order_date_clean#321]
            +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, amount#124, order_date#23, status#24]
               +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, CASE WHEN RLIKE(amount_norm#123, ^[+-]?\d+$) THEN cast(amount_norm#123 as int) ELSE cast(null as int

# Analytics

## Total revenue per city

In [35]:
from pyspark.sql import functions as F

revenue_by_city = (
    completed_df
    .groupBy("city")
    .agg(F.sum("amount").alias("total_revenue"))
)

revenue_by_city.show(50, truncate=False)


+---------+-------------+
|city     |total_revenue|
+---------+-------------+
|Bangalore|1628527093   |
|Chennai  |1629865247   |
|Mumbai   |1625518096   |
|Kolkata  |1624300497   |
|Pune     |1646196535   |
|Delhi    |1639639916   |
|Hyderabad|1642443340   |
+---------+-------------+



## Total revenue per category

In [36]:
revenue_by_category = (
    completed_df
    .groupBy("category")
    .agg(F.sum("amount").alias("total_revenue"))
)

revenue_by_category.show(50, truncate=False)


+-----------+-------------+
|category   |total_revenue|
+-----------+-------------+
|Home       |2868467576   |
|Fashion    |2834182172   |
|Grocery    |2866272106   |
|Electronics|2867568870   |
+-----------+-------------+



# Average order value (AOV) per city

In [38]:

aov_by_city = (
    completed_df
    .groupBy("city")
    .agg(F.avg("amount").alias("avg_order_value"))
)

aov_by_city.show(50, truncate=False)


+---------+------------------+
|city     |avg_order_value   |
+---------+------------------+
|Bangalore|44098.867908689645|
|Chennai  |43628.27900315863 |
|Mumbai   |43723.75651612556 |
|Kolkata  |43709.816662630175|
|Pune     |43930.204013556424|
|Delhi    |43817.20780331374 |
|Hyderabad|43708.74045293664 |
+---------+------------------+



In [39]:

aov_by_city = (
    completed_df
    .filter(F.col("amount").isNotNull())
    .groupBy("city")
    .agg((F.sum("amount") / F.count(F.lit(1))).alias("avg_order_value"))
)


# Window Functions

In [40]:
from pyspark.sql import functions as F
from pyspark.sql import Window


In [41]:
revenue_by_city = (
    completed_df
    .groupBy("city")
    .agg(F.sum("amount").alias("total_revenue"))
)


In [42]:
w_city = Window.orderBy(F.col("total_revenue").desc(), F.col("city").asc())

In [44]:

ranked_cities = (
    revenue_by_city
    .withColumn("revenue_rank", F.dense_rank().over(w_city))
    .orderBy(F.col("revenue_rank").asc(), F.col("city").asc())
)

ranked_cities.show(100, truncate=False)


+---------+-------------+------------+
|city     |total_revenue|revenue_rank|
+---------+-------------+------------+
|Pune     |1646196535   |1           |
|Hyderabad|1642443340   |2           |
|Delhi    |1639639916   |3           |
|Chennai  |1629865247   |4           |
|Bangalore|1628527093   |5           |
|Mumbai   |1625518096   |6           |
|Kolkata  |1624300497   |7           |
+---------+-------------+------------+



# Rank products inside each category by revenue

In [45]:

revenue_by_cat_prod = (
    completed_df
    .groupBy("category", "product")
    .agg(F.sum("amount").alias("total_revenue"))
)

w_cat_prod = Window.partitionBy("category") \
                   .orderBy(F.col("total_revenue").desc(), F.col("product").asc())

ranked_products_within_category = (
    revenue_by_cat_prod
    .withColumn("rank_in_category", F.dense_rank().over(w_cat_prod))
    .orderBy("category", "rank_in_category", "product")
)

ranked_products_within_category.show(200, truncate=False)


+-----------+-----------+-------------+----------------+
|category   |product    |total_revenue|rank_in_category|
+-----------+-----------+-------------+----------------+
|Electronics|Laptop     |962496295    |1               |
|Electronics|Tablet     |960719999    |2               |
|Electronics|Mobile     |944352576    |3               |
|Fashion    |Jeans      |951286127    |1               |
|Fashion    |Shoes      |946799102    |2               |
|Fashion    |Tshirt     |936096943    |3               |
|Grocery    |Oil        |963572869    |1               |
|Grocery    |Rice       |954494237    |2               |
|Grocery    |Sugar      |948205000    |3               |
|Home       |Vacuum     |959149427    |1               |
|Home       |Mixer      |957140026    |2               |
|Home       |Airpurifier|952178123    |3               |
+-----------+-----------+-------------+----------------+



# Find the top product for every category

In [46]:
top_product_per_category = (
    ranked_products_within_category
    .filter(F.col("rank_in_category") == 1)
    .orderBy("category", "product")
)

top_product_per_category.show(truncate=False)


+-----------+-------+-------------+----------------+
|category   |product|total_revenue|rank_in_category|
+-----------+-------+-------------+----------------+
|Electronics|Laptop |962496295    |1               |
|Fashion    |Jeans  |951286127    |1               |
|Grocery    |Oil    |963572869    |1               |
|Home       |Vacuum |959149427    |1               |
+-----------+-------+-------------+----------------+



In [47]:
w_top1 = Window.partitionBy("category") \
               .orderBy(F.col("total_revenue").desc(), F.col("product").asc())

top1_strict_per_category = (
    revenue_by_cat_prod
    .withColumn("rn", F.row_number().over(w_top1))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .orderBy("category")
)

top1_strict_per_category.show(truncate=False)


+-----------+-------+-------------+
|category   |product|total_revenue|
+-----------+-------+-------------+
|Electronics|Laptop |962496295    |
|Fashion    |Jeans  |951286127    |
|Grocery    |Oil    |963572869    |
|Home       |Vacuum |959149427    |
+-----------+-------+-------------+



# Identify the top 3 performing cities

In [48]:
top3_cities = (
    ranked_cities
    .filter(F.col("revenue_rank") <= 3)
    .orderBy("revenue_rank", "city")
)

top3_cities.show(truncate=False)


+---------+-------------+------------+
|city     |total_revenue|revenue_rank|
+---------+-------------+------------+
|Pune     |1646196535   |1           |
|Hyderabad|1642443340   |2           |
|Delhi    |1639639916   |3           |
+---------+-------------+------------+



In [49]:

w_city_rownum = Window.orderBy(F.col("total_revenue").desc(), F.col("city").asc())

top3_cities_exact = (
    revenue_by_city
    .withColumn("rn", F.row_number().over(w_city_rownum))
    .filter(F.col("rn") <= 3)
    .drop("rn")
    .orderBy(F.col("total_revenue").desc(), F.col("city").asc())
)

top3_cities_exact.show(truncate=False)


+---------+-------------+
|city     |total_revenue|
+---------+-------------+
|Pune     |1646196535   |
|Hyderabad|1642443340   |
|Delhi    |1639639916   |
+---------+-------------+



# Broadcast

In [50]:

from pyspark.sql import functions as F

city_region_rows = [
    ("Delhi", "North"),
    ("Mumbai", "West"),
    ("Bangalore", "South"),
    ("Hyderabad", "South"),
    ("Pune", "West"),
    ("Chennai", "South"),
    ("Kolkata", "East"),
]

city_region_df = spark.createDataFrame(city_region_rows, ["city", "region"])

city_region_df = city_region_df.withColumn("city", F.initcap(F.col("city")))


In [51]:

from pyspark.sql.functions import broadcast

orders_with_region = (
    completed_df.alias("o")
    .join(broadcast(city_region_df).alias("cr"), on=F.col("o.city") == F.col("cr.city"), how="left")
    .select("o.*", "cr.region")
)

orders_with_region.show(10, truncate=False)


+-----------+-----------+---------+-----------+-------+------+----------+---------+----------------+------+
|order_id   |customer_id|city     |category   |product|amount|order_date|status   |order_date_clean|region|
+-----------+-----------+---------+-----------+-------+------+----------+---------+----------------+------+
|ORD00000001|C000001    |Pune     |Grocery    |Sugar  |35430 |2024-01-02|Completed|2024-01-02      |West  |
|ORD00000007|C000007    |Pune     |Grocery    |Rice   |45362 |2024-01-08|Completed|2024-01-08      |West  |
|ORD00000008|C000008    |Bangalore|Fashion    |Jeans  |10563 |2024-01-09|Completed|2024-01-09      |South |
|ORD00000010|C000010    |Bangalore|Grocery    |Sugar  |66576 |2024-01-11|Completed|2024-01-11      |South |
|ORD00000011|C000011    |Kolkata  |Electronics|Tablet |50318 |12/01/2024|Completed|2024-01-12      |East  |
|ORD00000012|C000012    |Bangalore|Grocery    |Sugar  |84768 |2024-01-13|Completed|2024-01-13      |South |
|ORD00000014|C000014    |Mum

In [52]:
orders_with_region.explain(True)

== Parsed Logical Plan ==
'Project [o.*, 'cr.region]
+- Join LeftOuter, (city#107 = city#1916)
   :- SubqueryAlias o
   :  +- Filter (lower(trim(status#24, None)) = completed)
   :     +- Deduplicate [order_id#17]
   :        +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, amount#124, order_date#23, status#24, to_date(coalesce(try_to_timestamp(order_date#23, Some(yyyy-MM-dd), TimestampType, Some(Etc/UTC), false), try_to_timestamp(order_date#23, Some(dd/MM/yyyy), TimestampType, Some(Etc/UTC), false), try_to_timestamp(order_date#23, Some(yyyy/MM/dd), TimestampType, Some(Etc/UTC), false)), None, Some(Etc/UTC), true) AS order_date_clean#321]
   :           +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, amount#124, order_date#23, status#24]
   :              +- Project [order_id#17, customer_id#18, city#107, category#108, product#109, CASE WHEN RLIKE(amount_norm#123, ^[+-]?\d+$) THEN cast(amount_norm#123 as int) ELSE cast(null as 

# UDF

In [53]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def classify_amount(amount):
    if amount is None:
        return "Low"
    try:
        val = int(amount)
    except Exception:
        return "Low"
    if val >= 80000:
        return "High"
    elif val >= 40000:
        return "Medium"
    else:
        return "Low"

classify_amount_udf = F.udf(classify_amount, StringType())

orders_with_category = completed_df.withColumn(
    "order_value_category",
    classify_amount_udf(F.col("amount"))
)

orders_with_category.select("order_id", "amount", "order_value_category").show(20, truncate=False)


+-----------+------+--------------------+
|order_id   |amount|order_value_category|
+-----------+------+--------------------+
|ORD00000001|35430 |Low                 |
|ORD00000007|45362 |Medium              |
|ORD00000008|10563 |Low                 |
|ORD00000010|66576 |Medium              |
|ORD00000011|50318 |Medium              |
|ORD00000012|84768 |High                |
|ORD00000014|79469 |Medium              |
|ORD00000015|81018 |High                |
|ORD00000017|69582 |Medium              |
|ORD00000019|NULL  |Low                 |
|ORD00000022|48832 |Medium              |
|ORD00000023|12000 |Low                 |
|ORD00000024|18082 |Low                 |
|ORD00000025|58248 |Medium              |
|ORD00000028|70675 |Medium              |
|ORD00000030|52112 |Medium              |
|ORD00000031|51151 |Medium              |
|ORD00000032|75797 |Medium              |
|ORD00000034|40915 |Medium              |
|ORD00000036|29253 |Low                 |
+-----------+------+--------------

# Analyze the distribution (counts & percentages)

In [54]:

total_rows = orders_with_category.count()

distribution_df = (
    orders_with_category
    .groupBy("order_value_category")
    .agg(F.count(F.lit(1)).alias("count"))
    .withColumn("percentage", (F.col("count") / F.lit(total_rows) * 100.0))
    .orderBy(F.col("count").desc(), F.col("order_value_category").asc())
)

distribution_df.show(truncate=False)


+--------------------+------+-----------------+
|order_value_category|count |percentage       |
+--------------------+------+-----------------+
|Low                 |145699|51.12245614035088|
|Medium              |111365|39.07543859649123|
|High                |27936 |9.802105263157895|
+--------------------+------+-----------------+



# RDD

In [55]:
orders_rdd = completed_df.rdd

In [57]:
print(orders_rdd.take(3))

[Row(order_id='ORD00000001', customer_id='C000001', city='Pune', category='Grocery', product='Sugar', amount=35430, order_date='2024-01-02', status='Completed', order_date_clean=datetime.date(2024, 1, 2)), Row(order_id='ORD00000007', customer_id='C000007', city='Pune', category='Grocery', product='Rice', amount=45362, order_date='2024-01-08', status='Completed', order_date_clean=datetime.date(2024, 1, 8)), Row(order_id='ORD00000008', customer_id='C000008', city='Bangalore', category='Fashion', product='Jeans', amount=10563, order_date='2024-01-09', status='Completed', order_date_clean=datetime.date(2024, 1, 9))]


In [58]:

city_amount_rdd = (
    completed_df
    .select("city", "amount")
    .rdd
    .map(lambda r: (r["city"], r["amount"]))
    .filter(lambda x: x[1] is not None)
)


# Total revenue using reduce

In [60]:

amount_rdd = (
    completed_df
    .select("amount")
    .rdd
    .map(lambda r: r["amount"])
    .filter(lambda a: a is not None)
)

total_revenue = amount_rdd.reduce(lambda a, b: a + b) if not amount_rdd.isEmpty() else 0
print(f"Total revenue (RDD reduce): {total_revenue}")


Total revenue (RDD reduce): 11436490724


# Orders per city using map and reduce

In [61]:

orders_per_city_count = (
    completed_df
    .select("city")
    .rdd
    .map(lambda r: (r["city"], 1))
    .reduceByKey(lambda x, y: x + y)
)

print("Orders per city (count):")
for city, cnt in orders_per_city_count.collect():
    print(city, cnt)


Orders per city (count):
Pune 40883
Mumbai 40612
Hyderabad 41041
Delhi 40854
Bangalore 40311
Kolkata 40563
Chennai 40736


# Revenue per city (sum of amount per city)

In [62]:

revenue_per_city = (
    city_amount_rdd
    .map(lambda x: (x[0], x[1]))
    .reduceByKey(lambda a, b: a + b)
)

print("Revenue per city (sum of amount):")
for city, rev in revenue_per_city.collect():
    print(city, rev)

Revenue per city (sum of amount):
Pune 1646196535
Mumbai 1625518096
Hyderabad 1642443340
Delhi 1639639916
Bangalore 1628527093
Kolkata 1624300497
Chennai 1629865247


# Cache

In [63]:
completed_df.cache()
completed_df.count()


285000

# Compare

In [64]:
import time

start = time.time()
revenue_by_city.show()
print("Query 1 time:", time.time() - start)

start = time.time()
revenue_by_category.show()
print("Query 2 time:", time.time() - start)


+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|   1628527093|
|  Chennai|   1629865247|
|   Mumbai|   1625518096|
|  Kolkata|   1624300497|
|     Pune|   1646196535|
|    Delhi|   1639639916|
|Hyderabad|   1642443340|
+---------+-------------+

Query 1 time: 5.299536228179932
+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|       Home|   2868467576|
|    Fashion|   2834182172|
|    Grocery|   2866272106|
|Electronics|   2867568870|
+-----------+-------------+

Query 2 time: 4.481026649475098


In [65]:
completed_df.unpersist()

DataFrame[order_id: string, customer_id: string, city: string, category: string, product: string, amount: int, order_date: string, status: string, order_date_clean: date]

# Convert data to parquet and ORC

In [66]:

base_path = "/tmp/retail_case_study"
parquet_path = f"{base_path}/parquet/orders_by_city"
orc_base_path = f"{base_path}/orc"
orc_revenue_by_city_path = f"{orc_base_path}/revenue_by_city"
orc_revenue_by_category_path = f"{orc_base_path}/revenue_by_category"
orc_aov_by_city_path = f"{orc_base_path}/aov_by_city"
csv_path = f"{base_path}/csv/orders_clean"


In [67]:

from pyspark.sql import functions as F
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

completed_df_repart = completed_df.repartition("city")

(
    completed_df_repart
    .write
    .mode("overwrite")
    .partitionBy("city")
    .parquet(parquet_path)
)


spark.sql.shuffle.partitions: 200


In [69]:

from pyspark.sql import functions as F

orc_base_path = "/tmp/retail_case_study/orc"
orc_revenue_by_city_path = f"{orc_base_path}/revenue_by_city"
orc_revenue_by_category_path = f"{orc_base_path}/revenue_by_category"
orc_aov_by_city_path = f"{orc_base_path}/aov_by_city"

revenue_by_city = (
    completed_df
    .groupBy("city")
    .agg(F.sum("amount").alias("total_revenue"))
)

revenue_by_category = (
    completed_df
    .groupBy("category")
    .agg(F.sum("amount").alias("total_revenue"))
)

aov_by_city = (
    completed_df
    .groupBy("city")
    .agg(F.avg("amount").alias("avg_order_value"))
)

revenue_by_city.write.mode("overwrite").orc(orc_revenue_by_city_path)
revenue_by_category.write.mode("overwrite").orc(orc_revenue_by_category_path)
aov_by_city.write.mode("overwrite").orc(orc_aov_by_city_path)
